# Experimento principal

## Imports necesarios

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from joblib import Parallel, delayed

## Funciones 

In [ ]:
def procesar_dataset(archivo, n_splits=10, n_repeats=1):
    """
    Procesa el dataset usando SVM lineal.
    Soporta Validación Cruzada Clásica (1 repetición) o Repetida (N repeticiones).
    """
    if not os.path.exists(archivo):
        return archivo, None, None, f"[ERROR] Archivo no encontrado: {archivo}"

    df = pd.read_csv(archivo)
    
    # 1. Eliminar variables de sesgo global (Data Leakage)
    columnas_sesgo = ['P_0', 'P_1', 'P_2', 'mu_2', 'sigma_2']
    cols_a_borrar = ['clase'] + [c for c in columnas_sesgo if c in df.columns]
    
    X = df.drop(columns=cols_a_borrar).values
    y = df['clase'].values

    metricas = {
        'acc_train': [], 
        'acc_test': [], 
        'recall_test': [], 
        'f1_test': [], 
        'auc_test': []
    }
    
    # 2. Configurar la Validación Cruzada Estratificada Repetida
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

    fold = 1
    for train_idx, test_idx in rskf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        svm = SVC(kernel='linear', random_state=42)
        svm.fit(X_train, y_train)

        y_pred_train = svm.predict(X_train)
        y_pred_test = svm.predict(X_test)
        y_scores_test = svm.decision_function(X_test)

        metricas['acc_train'].append(accuracy_score(y_train, y_pred_train))
        metricas['acc_test'].append(accuracy_score(y_test, y_pred_test))
        metricas['recall_test'].append(recall_score(y_test, y_pred_test))
        metricas['f1_test'].append(f1_score(y_test, y_pred_test))
        metricas['auc_test'].append(roc_auc_score(y_test, y_scores_test))

        fold += 1

    # 3. Empaquetar resultados
    total_iteraciones = n_splits * n_repeats
    df_metricas = pd.DataFrame(metricas)
    df_metricas.index = [f"Iteracion_{i+1}" for i in range(total_iteraciones)]
    
    # Añadir el nombre del esquema para facilitar la combinación posterior
    nombre_esquema = archivo.replace('.csv', '').replace('radiomics_', '')
    df_metricas.insert(0, 'Esquema', nombre_esquema)

    stats = pd.DataFrame({
        'Media': df_metricas.iloc[:, 1:].mean(),
        'Mediana': df_metricas.iloc[:, 1:].median(),
        'Desv. Est.': df_metricas.iloc[:, 1:].std()
    }).T

    return archivo, df_metricas, stats, None

def ejecutar_experimento(nombre_experimento, archivos, n_splits, n_repeats):
    """Orquesta la ejecución paralela y guarda los CSVs resultantes."""
    total_folds = n_splits * n_repeats
    print(f"\n{'='*80}")
    print(f"[INICIO] Experimento: {nombre_experimento} ({total_folds} Folds Totales)")
    print(f"{'='*80}")
    
    resultados = Parallel(n_jobs=-1)(delayed(procesar_dataset)(archivo, n_splits, n_repeats) for archivo in archivos)

    resumen_medias = []
    lista_metricas_crudas = []

    for archivo, df_metricas, stats, error in resultados:
        if error:
            print(error)
            continue
            
        lista_metricas_crudas.append(df_metricas)

        media_esquema = stats.loc['Media'].copy()
        nombre_limpio = archivo.replace('.csv', '').replace('radiomics_', '')
        media_esquema['Esquema'] = nombre_limpio
        resumen_medias.append(media_esquema)

    if resumen_medias:
        # 1. Guardar y mostrar tabla resumen comparativa
        df_final = pd.DataFrame(resumen_medias)
        df_final.set_index('Esquema', inplace=True)
        df_final = df_final.sort_values(by='acc_test', ascending=False)
        
        archivo_resumen = f"resumen_{nombre_experimento}_{total_folds}folds.csv"
        df_final.to_csv(archivo_resumen)
        
        print(f"\n[RESULTADO] Resumen guardado en: {archivo_resumen}")
        print(df_final.round(4).to_string())
        
        # 2. Guardar métricas crudas (necesarias para el test estadístico de mañana)
        df_crudos_total = pd.concat(lista_metricas_crudas)
        archivo_crudos = f"metricas_crudas_{nombre_experimento}_{total_folds}folds.csv"
        df_crudos_total.to_csv(archivo_crudos, index=False)
        print(f"[INFO] Métricas detalladas guardadas en: {archivo_crudos}")

if __name__ == "__main__":
    archivos_datasets = [
        "radiomics_DE_rand_1.csv",
        "radiomics_DE_best_1.csv",
        "radiomics_DE_rand_2.csv",
        "radiomics_DE_best_2.csv",
        "radiomics_Standard_Otsu.csv"
    ]

    print("[INFO] Iniciando Pipeline de Evaluación SVM (Ignorando variables de sesgo)")

    # EXPERIMENTO 1: 10 Folds Clásicos (1 repetición)
    ejecutar_experimento(
        nombre_experimento="Clasico", 
        archivos=archivos_datasets, 
        n_splits=10, 
        n_repeats=1
    )

    # EXPERIMENTO 2: 30 Folds Repetidos (10 splits x 3 repeticiones)
    ejecutar_experimento(
        nombre_experimento="Repetido", 
        archivos=archivos_datasets, 
        n_splits=10, 
        n_repeats=3
    )
    
    print("\n[FIN] Todos los experimentos han concluido y los resultados se han guardado exitosamente.")